In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from go_search_problem import GoProblem, GoState
from game_runner import GameRunner
import matplotlib.pyplot as plt
import pickle
from collections import deque
import itertools
from typing import List, Tuple, Dict
from open_spiel.python.rl_environment import Environment



def get_features(game_state: GoState):
    """
    Map a game state to a list of features.

    Some useful functions from game_state include:
        game_state.size: size of the board
        get_pieces_coordinates(player_index): get coordinates of all pieces of a player (0 or 1)
        get_pieces_array(player_index): get a 2D array of pieces of a player (0 or 1)
        
        get_board(): get a 2D array of the board with 4 channels (player 0, player 1, empty, and player to move). 4 channels means the array will be of size 4 x n x n
    
        Descriptions of these methods can be found in the GoState

    Input:
        game_state: GoState to encode into a fixed size list of features
    Output:
        features: list of features
    """

    board_size = game_state.size
    # TODO: Encode game_state into a list of features
    features = []
    board = game_state.get_board()
    black_pieces = board[0].reshape(board_size * board_size)
    white_pieces = board[1].reshape(board_size * board_size)
    player = game_state.player_to_move()

    # stone difference
    stone_diff = np.sum(black_pieces) - np.sum(white_pieces)


    #legal actions
    legal = np.zeros(26)
    for action in game_state.legal_actions():
        legal[action] = 1

    features = np.concatenate((black_pieces, white_pieces, [player], [stone_diff], legal))

    return features




class DQN(nn.Module):
    def __init__(self, input_size, output_size):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.output_layer = nn.Linear(64, output_size)
        self.relu = nn.ReLU()


    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output_layer(x)
    



def train_dqn():
    

# Features
Get features for network

# Defining q network

In [ ]:
class DQN(nn.Module):
    def __init__(self, input_size, output_size):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.output_layer = nn.Linear(64, output_size)
        self.relu = nn.ReLU()


    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output_layer(x)

In [ ]:
env = Environment("go", board_size=5)

gamma = 0.99
epsilon = 1
epsilon_min = 0.01
epsilon_decay = 0.995
learning_rate = 0.001
memory_size = 10000
batch_size = 64


memory = deque(maxlen=memory_size)

In [ ]:
def q_learning(mdp: GoProblem, num_episodes: int, alpha: float) -> Tuple[QTable, Dict[Tuple[GoState, GoAction], int]]:


    observation_counts = {} 
    actions = (0, 1)
    states = mdp.get_all_states()
    epsilon = 0.2

    # Initialize Q-table and observation counts
    for state_action_pair in list(itertools.product(states, actions)):
        Q[state_action_pair] = 0.0
        observation_counts[state_action_pair] = 0 

    # TODO: Implement Q-Learning
    for _ in range(num_episodes):
        done = False
        state = mdp.reset()

        while not done:
            if np.random.rand() < epsilon:
                action = np.random.choice(actions)
            else:
                q_values = [Q[(state, a)] for a in actions]
                max_q_value = max(q_values)
                best_actions = [a for a in actions if Q[(state, a)] == max_q_value]
                action = np.random.choice(best_actions)

            next_state, reward, done = mdp.step(action)
            observation_counts[(state, action)] += 1


            if done:
                best_next_q_value = 0
            else:
                best_next_q_value = max(Q.get((next_state, a), 0) for a in actions)

            Q[(state, action)] = (1 - alpha) * Q[(state, action)] + alpha * (reward + best_next_q_value)

            state = next_state


    return Q, observation_counts